In [1]:
# Cell 1: Imports and configuration
import os
from pathlib import Path
import libsbml
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [16]:
# Cell 2: Deep SBML structure exploration

def explore_sbml_structure(sbml_file):
    """
    Explore ALL possible locations where gene information might be stored in an SBML file.
    Returns a dictionary with findings from different sections.
    
    Parameters:
    -----------
    sbml_file : str
        Path to SBML file
    
    Returns:
    --------
    dict
        Dictionary containing all found structures and their counts
    """
    reader = libsbml.SBMLReader()
    document = reader.readSBML(sbml_file)
    
    model = document.getModel()
    if model is None:
        return {"error": "Could not parse model"}
    
    findings = {
        "model_id": model.getId() if model.isSetId() else "No ID",
        "model_name": model.getName() if model.isSetName() else "No name",
        "sbml_level": document.getLevel(),
        "sbml_version": document.getVersion(),
        "num_reactions": model.getNumReactions(),
        "num_species": model.getNumSpecies(),
        "num_compartments": model.getNumCompartments(),
    }
    
    # Check for FBC plugin and gene products
    try:
        fbc_plugin = model.getPlugin('fbc')
        if fbc_plugin:
            findings["fbc_plugin"] = True
            findings["num_gene_products"] = fbc_plugin.getNumGeneProducts()
            
            # Sample some gene product IDs
            sample_genes = []
            for i in range(min(5, fbc_plugin.getNumGeneProducts())):
                gp = fbc_plugin.getGeneProduct(i)
                sample_genes.append({
                    "id": gp.getId(),
                    "label": gp.getLabel() if gp.isSetLabel() else "No label",
                    "name": gp.getName() if gp.isSetName() else "No name"
                })
            findings["sample_gene_products"] = sample_genes
        else:
            findings["fbc_plugin"] = False
    except:
        findings["fbc_plugin"] = "Error accessing FBC"
    
    # Check reactions for gene associations
    gene_association_types = set()
    sample_associations = []
    
    for i in range(min(10, model.getNumReactions())):  # Check first 10 reactions
        reaction = model.getReaction(i)
        
        # Check FBC gene product association
        try:
            rxn_fbc = reaction.getPlugin('fbc')
            if rxn_fbc:
                gpa = rxn_fbc.getGeneProductAssociation()
                if gpa:
                    gene_association_types.add("FBC_GeneProductAssociation")
                    assoc = gpa.getAssociation()
                    if assoc:
                        sample_associations.append({
                            "reaction": reaction.getId(),
                            "type": "FBC_GPA",
                            "content": str(assoc.toInfix()) if hasattr(assoc, 'toInfix') else "Complex object"
                        })
        except:
            pass
        
        # Check notes
        notes = reaction.getNotesString()
        if notes and "gene" in notes.lower():
            gene_association_types.add("Notes_contains_gene")
            if len(sample_associations) < 3:
                sample_associations.append({
                    "reaction": reaction.getId(),
                    "type": "Notes",
                    "content": notes[:200]  # First 200 chars
                })
        
        # Check annotations
        annotation = reaction.getAnnotationString()
        if annotation and "gene" in annotation.lower():
            gene_association_types.add("Annotation_contains_gene")
            if len(sample_associations) < 3:
                sample_associations.append({
                    "reaction": reaction.getId(),
                    "type": "Annotation",
                    "content": annotation[:200]
                })
    
    findings["gene_association_types"] = list(gene_association_types)
    findings["sample_associations"] = sample_associations
    
    # Check for any plugins
    plugins = []
    try:
        for i in range(model.getNumPlugins()):
            plugin = model.getPlugin(i)
            plugins.append(plugin.getPackageName())
    except:
        pass
    findings["plugins"] = plugins
    
    return findings

print("✓ Exploration function defined")

✓ Exploration function defined


In [18]:
# Cell 3: Set path and explore ALL models

# CHANGE THIS PATH
base_path = Path("models/")

# Store all findings organized by method and organism
all_findings = {}

for method_folder in base_path.iterdir():
    if not method_folder.is_dir():
        continue
    
    method_name = method_folder.name
    all_findings[method_name] = {}
    
    print(f"\n{'='*80}")
    print(f"METHOD: {method_name}")
    print(f"{'='*80}")
    
    for sbml_file in method_folder.glob("*.xml"):
        organism = sbml_file.stem
        
        print(f"\n  Organism: {organism}")
        print(f"  File: {sbml_file.name}")
        print(f"  {'-'*76}")
        
        findings = explore_sbml_structure(str(sbml_file))
        all_findings[method_name][organism] = findings
        
        # Print key findings
        if "error" in findings:
            print(f"    ❌ {findings['error']}")
            continue
        
        print(f"    SBML Level {findings['sbml_level']}, Version {findings['sbml_version']}")
        print(f"    Reactions: {findings['num_reactions']}")
        print(f"    Species: {findings['num_species']}")
        print(f"    Plugins: {findings['plugins']}")
        print(f"    FBC Plugin: {findings['fbc_plugin']}")
        
        if findings.get('num_gene_products', 0) > 0:
            print(f"    ✓ Gene Products (FBC): {findings['num_gene_products']}")
            if findings.get('sample_gene_products'):
                print(f"      Sample IDs:")
                for gp in findings['sample_gene_products'][:3]:
                    print(f"        - {gp['id']} (label: {gp['label']})")
        else:
            print(f"    ✗ No FBC Gene Products found")
        
        if findings.get('gene_association_types'):
            print(f"    Gene Association Types Found: {findings['gene_association_types']}")
            if findings.get('sample_associations'):
                print(f"      Examples:")
                for assoc in findings['sample_associations'][:2]:
                    print(f"        [{assoc['type']}] {assoc['reaction']}")
                    print(f"          {assoc['content'][:100]}...")

print(f"\n{'='*80}")
print("✓ Exploration complete")


METHOD: carveMe

  Organism: E_siliculosus
  File: E_siliculosus.xml
  ----------------------------------------------------------------------------
    SBML Level 3, Version 1
    Reactions: 1548
    Species: 1080
    Plugins: ['fbc']
    FBC Plugin: True
    ✓ Gene Products (FBC): 786
      Sample IDs:
        - G_CBJ27061_1 (label: G_CBJ27061_1)
        - G_CBN77946_1 (label: G_CBN77946_1)
        - G_CBN76757_1 (label: G_CBN76757_1)
    Gene Association Types Found: ['FBC_GeneProductAssociation']
      Examples:
        [FBC_GPA] R_24DECOAR
          G_CBJ27061_1...

  Organism: CHO
  File: CHO.xml
  ----------------------------------------------------------------------------
    SBML Level 3, Version 1
    Reactions: 1163
    Species: 850
    Plugins: ['fbc']
    FBC Plugin: True
    ✓ Gene Products (FBC): 702
      Sample IDs:
        - G_EGW07470_1 (label: G_EGW07470_1)
        - G_EGW07730_1 (label: G_EGW07730_1)
        - G_EGW00123_1 (label: G_EGW00123_1)

  Organism: A_aegyp

In [19]:
# Cell 4: Summary table of what we found

summary_data = []

for method, organisms in all_findings.items():
    for organism, findings in organisms.items():
        if "error" not in findings:
            summary_data.append({
                'Method': method,
                'Organism': organism,
                'FBC_Plugin': findings.get('fbc_plugin', False),
                'Num_Gene_Products': findings.get('num_gene_products', 0),
                'Gene_Assoc_Types': ', '.join(findings.get('gene_association_types', [])),
                'Num_Reactions': findings.get('num_reactions', 0),
                'SBML_Level': findings.get('sbml_level', '?'),
                'Plugins': ', '.join(findings.get('plugins', []))
            })

df_summary = pd.DataFrame(summary_data)

print("\n📊 SUMMARY OF ALL MODELS")
print("="*80)
display(df_summary)

print("\n📈 Gene information availability by method:")
display(df_summary.groupby('Method').agg({
    'FBC_Plugin': 'sum',
    'Num_Gene_Products': 'mean',
    'Gene_Assoc_Types': lambda x: len(set(y for y in x if y))
}))


📊 SUMMARY OF ALL MODELS


,Method,Organism,FBC_Plugin,Num_Gene_Products,Gene_Assoc_Types,Num_Reactions,SBML_Level,Plugins
0,carveMe,E_siliculosus,True,786,FBC_GeneProductAssociation,1548,3,fbc
1,carveMe,CHO,True,702,,1163,3,fbc
2,carveMe,A_aegypti,True,649,,1087,3,fbc
3,merlin,E_siliculosus,True,1642,FBC_GeneProductAssociation,3813,3,"fbc, groups"
4,merlin,CHO,True,2807,FBC_GeneProductAssociation,6390,3,"fbc, groups"
5,merlin,A_aegypti,True,4306,FBC_GeneProductAssociation,4820,3,"fbc, groups"
6,modelseed,E_siliculosus,False,0,Notes_contains_gene,1104,2,layout
7,modelseed,CHO,False,0,Notes_contains_gene,1104,2,layout
8,modelseed,A_aegypti,False,0,Notes_contains_gene,1104,2,layout
9,raven_homo,E_siliculosus,True,1141,FBC_GeneProductAssociation,1710,3,"fbc, groups"



📈 Gene information availability by method:


,FBC_Plugin,Num_Gene_Products,Gene_Assoc_Types
Method,,,
AuReMe,3,0.000000,1
carveMe,3,712.333333,1
merlin,3,2918.333333,1
modelseed,0,0.000000,1
pathway_tools,3,0.000000,1
raven,3,2460.000000,1
raven_comb,3,3037.666667,1
raven_homo,3,2194.333333,1
reference,3,1077.333333,2


In [20]:
# Cell 5: Deep dive into a specific problematic model (like AuReMe)

# CHANGE THESE
problem_method = "AuReMe"  # or whatever method has issues
problem_organism = "organism_name"

if problem_method in all_findings and problem_organism in all_findings[problem_method]:
    findings = all_findings[problem_method][problem_organism]
    
    print(f"🔍 DEEP DIVE: {problem_method}/{problem_organism}")
    print("="*80)
    
    # Show everything we found
    for key, value in findings.items():
        print(f"\n{key}:")
        if isinstance(value, list):
            for item in value:
                print(f"  - {item}")
        else:
            print(f"  {value}")
    
    # Now let's look at the RAW XML to see what's really there
    sbml_file = base_path / problem_method / f"{problem_organism}.xml"
    
    print(f"\n\n{'='*80}")
    print("RAW XML INSPECTION (first 5000 chars)")
    print("="*80)
    
    with open(sbml_file, 'r', encoding='utf-8') as f:
        content = f.read(5000)
        print(content)

In [13]:
# Cell 7: Compare gene overlap between methods

comparison_data = []

# Get all unique organisms across methods
all_organisms = set()
for method_data in results.values():
    all_organisms.update(method_data.keys())

# Compare methods pairwise for each organism
for organism in sorted(all_organisms):
    # Collect genes from each method for this organism
    methods_genes = {}
    for method, organisms_dict in results.items():
        if organism in organisms_dict:
            methods_genes[method] = organisms_dict[organism]
    
    # Calculate pairwise intersections between methods
    methods_list = list(methods_genes.keys())
    for i, method1 in enumerate(methods_list):
        for method2 in methods_list[i+1:]:
            genes1 = methods_genes[method1]
            genes2 = methods_genes[method2]
            
            # Set operations for comparison
            intersection = genes1 & genes2
            union = genes1 | genes2
            only_method1 = genes1 - genes2
            only_method2 = genes2 - genes1
            
            # Calculate similarity metrics
            jaccard = len(intersection) / len(union) if len(union) > 0 else 0
            overlap_pct = (len(intersection) / min(len(genes1), len(genes2)) * 100) if min(len(genes1), len(genes2)) > 0 else 0
            
            comparison_data.append({
                'Organism': organism,
                'Method_1': method1,
                'Method_2': method2,
                'Genes_Method_1': len(genes1),
                'Genes_Method_2': len(genes2),
                'Shared_Genes': len(intersection),
                'Only_Method_1': len(only_method1),
                'Only_Method_2': len(only_method2),
                'Jaccard_Index': jaccard,
                'Overlap_Percent': overlap_pct
            })

df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.sort_values(['Organism', 'Jaccard_Index'], ascending=[True, False])

print("🔬 METHOD COMPARISON")
print("=" * 60)
display(df_comparison)

🔬 METHOD COMPARISON


,Organism,Method_1,Method_2,Genes_Method_1,Genes_Method_2,Shared_Genes,Only_Method_1,Only_Method_2,Jaccard_Index,Overlap_Percent
27,A_aegypti,raven,raven_comb,3677,4604,3677,0,927,0.798653,100.000000
23,A_aegypti,raven_homo,raven_comb,3452,4604,3452,0,1152,0.749783,100.000000
21,A_aegypti,raven_homo,raven,3452,3677,2525,927,1152,0.548436,73.146002
0,A_aegypti,carveMe,merlin,649,4306,421,228,3885,0.092854,64.869029
1,A_aegypti,carveMe,modelseed,649,0,0,649,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
103,E_siliculosus,AuReMe,pathway_tools,0,0,0,0,0,0.000000,0.000000
104,E_siliculosus,AuReMe,reference,0,0,0,0,0,0.000000,0.000000
105,E_siliculosus,raven_comb,pathway_tools,1700,0,0,1700,0,0.000000,0.000000
106,E_siliculosus,raven_comb,reference,1700,0,0,1700,0,0.000000,0.000000


In [23]:
# Cell 3: Explore ALL models - ORGANIZED BY ORGANISM
from collections import defaultdict
# CHANGE THIS PATH
base_path = Path("models/")

# Store all findings organized by ORGANISM first, then method
all_findings_by_organism = defaultdict(dict)

# First pass: collect all data
for method_folder in base_path.iterdir():
    if not method_folder.is_dir():
        continue
    
    method_name = method_folder.name
    
    for sbml_file in method_folder.glob("*.xml"):
        organism = sbml_file.stem
        findings = explore_sbml_structure(str(sbml_file))
        all_findings_by_organism[organism][method_name] = findings

# Second pass: print organized by organism
for organism in sorted(all_findings_by_organism.keys()):
    print(f"\n{'='*80}")
    print(f"🧬 ORGANISM: {organism}")
    print(f"{'='*80}")
    
    methods_data = all_findings_by_organism[organism]
    
    for method_name, findings in methods_data.items():
        print(f"\n  📁 Method: {method_name}")
        print(f"  {'-'*76}")
        
        # Print key findings
        if "error" in findings:
            print(f"    ❌ {findings['error']}")
            continue
        
        print(f"    SBML Level {findings['sbml_level']}, Version {findings['sbml_version']}")
        print(f"    Reactions: {findings['num_reactions']}, Species: {findings['num_species']}")
        print(f"    Plugins: {findings['plugins']}")
        print(f"    FBC Plugin: {findings['fbc_plugin']}")
        
        if findings.get('num_gene_products', 0) > 0:
            print(f"    ✓ Gene Products (FBC): {findings['num_gene_products']}")
            if findings.get('sample_gene_products'):
                print(f"      Sample Gene IDs:")
                for gp in findings['sample_gene_products'][:3]:
                    print(f"        - ID: {gp['id']}")
                    if gp['label'] != "No label":
                        print(f"          Label: {gp['label']}")
                    if gp['name'] != "No name":
                        print(f"          Name: {gp['name']}")
        else:
            print(f"    ✗ No FBC Gene Products found")
        
        if findings.get('gene_association_types'):
            print(f"    Gene Association Types: {findings['gene_association_types']}")
            if findings.get('sample_associations'):
                print(f"      Examples:")
                for assoc in findings['sample_associations'][:2]:
                    print(f"        [{assoc['type']}] Reaction: {assoc['reaction']}")
                    print(f"          {assoc['content'][:150]}...")
    
    # Summary comparison for this organism
    print(f"\n  {'─'*76}")
    print(f"  📊 Summary for {organism}:")
    has_genes = {}
    for method_name, findings in methods_data.items():
        if "error" not in findings:
            num_genes = findings.get('num_gene_products', 0)
            has_genes[method_name] = num_genes
            gene_types = findings.get('gene_association_types', [])
            status = "✓" if num_genes > 0 or gene_types else "✗"
            print(f"    {status} {method_name}: {num_genes} FBC genes, Types: {gene_types}")
    print(f"  {'─'*76}")

print(f"\n{'='*80}")
print("✓ Exploration complete")


🧬 ORGANISM: A_aegypti

  📁 Method: carveMe
  ----------------------------------------------------------------------------
    SBML Level 3, Version 1
    Reactions: 1087, Species: 790
    Plugins: ['fbc']
    FBC Plugin: True
    ✓ Gene Products (FBC): 649
      Sample Gene IDs:
        - ID: G_XP_001649088_2
          Label: G_XP_001649088_2
          Name: G_XP_001649088_2
        - ID: G_XP_021696045_1
          Label: G_XP_021696045_1
          Name: G_XP_021696045_1
        - ID: G_XP_001662725_1
          Label: G_XP_001662725_1
          Name: G_XP_001662725_1

  📁 Method: merlin
  ----------------------------------------------------------------------------
    SBML Level 3, Version 2
    Reactions: 4820, Species: 4466
    Plugins: ['fbc', 'groups']
    FBC Plugin: True
    ✓ Gene Products (FBC): 4306
      Sample Gene IDs:
        - ID: G_XP_021702256_1
          Label: XP_021702256_1
        - ID: G_XP_021698884_1
          Label: XP_021698884_1
        - ID: G_XP_021702715_1